<a href="https://colab.research.google.com/github/koushik0728/RAG-with-reranking/blob/main/RAG_with_reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Install necessary libraries
!pip install -qqq datasets sentence-transformers faiss-cpu rank_bm25 transformers

In [8]:
# Import libraries
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
# Load a dataset (e.g., SQuAD v2 for contexts)
dataset = load_dataset("rajpurkar/squad_v2", split="train")

# Convert to Pandas DataFrame for easier manipulation (optional but often helpful)
df = pd.DataFrame(dataset)

# Display basic info about the dataset
print(f"Dataset loaded with {len(df)} examples.")
print("Columns in the dataset:")
print(df.columns)

# Display the first few rows and the 'context' column
print("\nFirst 5 rows of the dataset:")
print(df[['context', 'question', 'answers']].head())

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Dataset loaded with 130319 examples.
Columns in the dataset:
Index(['id', 'title', 'context', 'question', 'answers'], dtype='object')

First 5 rows of the dataset:
                                             context  \
0  Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...   
1  Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...   
2  Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...   
3  Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...   
4  Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...   

                                            question  \
0           When did Beyonce start becoming popular?   
1  What areas did Beyonce compete in when she was...   
2  When did Beyonce leave Destiny's Child and bec...   
3      In what city and state did Beyonce  grow up?    
4         In which decade did Beyonce become famous?   

                                             answers  
0  {'text': ['in the late 1990s'], 'answer_start'...  
1  {'text': ['singing and dancing'], 'answer_star...

In [10]:
# Extract unique contexts to form our document corpus
documents = df['context'].unique().tolist()

print(f"Extracted {len(documents)} unique documents.")
print("\nFirst document example:")
print(documents[0][:500]) # Print first 500 characters of the first document

Extracted 19029 unique documents.

First document example:
Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut al


In [11]:
# Load a pre-trained sentence transformer model
# This model will convert text into numerical vectors (embeddings)
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

print(f"Embedding model '{embedding_model_name}' loaded.")

# Generate embeddings for all documents
print(f"Generating embeddings for {len(documents)} documents. This may take a moment...")
document_embeddings = embedder.encode(documents, show_progress_bar=True)

print(f"Embeddings generated. Shape: {document_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model 'sentence-transformers/all-MiniLM-L6-v2' loaded.
Generating embeddings for 19029 documents. This may take a moment...


Batches:   0%|          | 0/595 [00:00<?, ?it/s]

Embeddings generated. Shape: (19029, 384)


In [12]:
# Create a FAISS index for efficient similarity search
dimension = document_embeddings.shape[1] # Dimension of the embeddings
index = faiss.IndexFlatL2(dimension) # Using L2 distance for similarity

# Add the document embeddings to the index
index.add(document_embeddings)

print(f"FAISS index created with {index.ntotal} documents.")
print("FAISS index is ready for similarity search.")

FAISS index created with 19029 documents.
FAISS index is ready for similarity search.


In [13]:
def retrieve_candidate_documents(query: str, top_k: int = 5) -> list[str]:
    """
    Retrieves candidate documents using vector similarity search.

    Args:
        query (str): The search query.
        top_k (int): The number of top documents to retrieve.

    Returns:
        list[str]: A list of retrieved document texts.
    """
    # Encode the query into a vector
    query_embedding = embedder.encode([query])

    # Perform a similarity search on the FAISS index
    # D: distances, I: indices of the top_k documents
    distances, indices = index.search(query_embedding, top_k)

    # Retrieve the actual documents using the indices
    retrieved_docs = [documents[idx] for idx in indices[0]]

    return retrieved_docs

# Example usage of the retrieval function
example_query = "When was the internet invented?"
top_k_retrieval = 5
candidate_documents = retrieve_candidate_documents(example_query, top_k_retrieval)

print(f"Query: {example_query}")
print(f"\nTop {top_k_retrieval} Candidate Documents (without reranking):")
for i, doc in enumerate(candidate_documents):
    print(f"--- Document {i+1} ---")
    print(doc[:200] + "...") # Print first 200 characters of each document

Query: When was the internet invented?

Top 5 Candidate Documents (without reranking):
--- Document 1 ---
The Internet was developed as a network between government research laboratories and participating departments of universities. By the late 1980s, a process was set in place towards public, commercial...
--- Document 2 ---
In 1988, only 60,000 computers were connected to the Internet, and most were mainframes, minicomputers and professional workstations. On November 2, 1988, many started to slow down, because they were ...
--- Document 3 ---
In time, the network spread beyond academic and military institutions and became known as the Internet. The emergence of networking involved a redefinition of the nature and boundaries of the computer...
--- Document 4 ---
Somalia established its first ISP in 1999, one of the last countries in Africa to get connected to the Internet. According to the telecommunications resource Balancing Act, growth in internet connecti...
--- Document 5 ---
Th

In [14]:
# Load a cross-encoder model for reranking
reranker_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)
model = AutoModelForSequenceClassification.from_pretrained(reranker_model_name)

# Create a Hugging Face pipeline for easier inference
reranker = pipeline("text-classification", model=model, tokenizer=tokenizer)

print(f"Reranking model '{reranker_model_name}' loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranking model 'cross-encoder/ms-marco-MiniLM-L-6-v2' loaded.


In [15]:
def rerank_documents(query: str, candidate_docs: list[str]) -> list[tuple[str, float]]:
    """
    Reranks candidate documents based on relevance to the query using a cross-encoder.

    Args:
        query (str): The search query.
        candidate_docs (list[str]): A list of documents to be reranked.

    Returns:
        list[tuple[str, float]]: A list of (document, relevance_score) tuples, sorted by score.
    """
    if not candidate_docs:
        return []

    # Prepare input for the reranker: pairs of (query, document)
    features = []
    for doc in candidate_docs:
        features.append(f"{query} [SEP] {doc}") # Common format for cross-encoders

    # Get relevance scores from the reranker
    # The model outputs logits, which represent relevance. We'll use them directly.
    results = reranker(features)

    # Pair documents with their scores and sort in descending order of score
    reranked_docs_with_scores = []
    for i, doc in enumerate(candidate_docs):
        # The 'score' key in the result dictionary typically represents the relevance score
        # For binary classification (relevant/not relevant), 'score' is usually for the positive class.
        reranked_docs_with_scores.append((doc, results[i]['score']))

    # Sort by score in descending order
    reranked_docs_with_scores.sort(key=lambda x: x[1], reverse=True)

    return reranked_docs_with_scores

# Example usage of the reranking function
reranked_documents = rerank_documents(example_query, candidate_documents)

print(f"\nTop {top_k_retrieval} Reranked Documents (with scores) for query: '{example_query}':")
for i, (doc, score) in enumerate(reranked_documents):
    print(f"--- Document {i+1} (Score: {score:.4f}) ---")
    print(doc[:200] + "...") # Print first 200 characters of each document


Top 5 Reranked Documents (with scores) for query: 'When was the internet invented?':
--- Document 1 (Score: 0.3822) ---
In 1988, only 60,000 computers were connected to the Internet, and most were mainframes, minicomputers and professional workstations. On November 2, 1988, many started to slow down, because they were ...
--- Document 2 (Score: 0.2915) ---
The first web browser was invented in 1990 by Sir Tim Berners-Lee. Berners-Lee is the director of the World Wide Web Consortium (W3C), which oversees the Web's continued development, and is also the f...
--- Document 3 (Score: 0.1565) ---
The Internet was developed as a network between government research laboratories and participating departments of universities. By the late 1980s, a process was set in place towards public, commercial...
--- Document 4 (Score: 0.0190) ---
In time, the network spread beyond academic and military institutions and became known as the Internet. The emergence of networking involved a redefinition of the

In [16]:
print(f"\n--- Original Candidate Documents (Vector Similarity) for Query: '{example_query}' ---")
for i, doc in enumerate(candidate_documents):
    print(f"Document {i+1}:")
    print(doc[:200] + "...")
    print("\n")

print(f"\n--- Reranked Documents (Cross-Encoder Scores) for Query: '{example_query}' ---")
for i, (doc, score) in enumerate(reranked_documents):
    print(f"Document {i+1} (Score: {score:.4f}):")
    print(doc[:200] + "...")
    print("\n")


--- Original Candidate Documents (Vector Similarity) for Query: 'When was the internet invented?' ---
Document 1:
The Internet was developed as a network between government research laboratories and participating departments of universities. By the late 1980s, a process was set in place towards public, commercial...


Document 2:
In 1988, only 60,000 computers were connected to the Internet, and most were mainframes, minicomputers and professional workstations. On November 2, 1988, many started to slow down, because they were ...


Document 3:
In time, the network spread beyond academic and military institutions and became known as the Internet. The emergence of networking involved a redefinition of the nature and boundaries of the computer...


Document 4:
Somalia established its first ISP in 1999, one of the last countries in Africa to get connected to the Internet. According to the telecommunications resource Balancing Act, growth in internet connecti...


Document 5:
The first web